# Check 01 — Core Orchestrator

**Category:** Module smoke check (fast regression; companion to pytest, not a full tutorial).

**Purpose:** Prove `Orchestrator.run_turn` routes a HIGH-risk, state-changing `planned_tool_call` through the deterministic tool path and emits `RUN_COMPLETE` plus completed `TOOL_PROGRESS`.

**Prerequisites:**
- Python **3.12+** with project deps installed (`pip install -r requirements.txt` from repo root)
- Kernel: project **`.venv`** (see `notebooks/README.md`)
- Run cells **top to bottom** (bootstrap cell sets `sys.path` automatically)
- **No API key** required — deterministic, in-process only

**Related tutorial:** `tutorial_01_core_framework.ipynb`

**Modules exercised:** `src/core/orchestrator`, `src/policies/middleware`, `src/runtime/openai_agents_runtime`, `src/tools/executor`, `src/tools/registry`

**PASS means:** Printed event stream includes `run_complete` and `tool_progress` with `state=completed`; final line is `PASS: orchestrator deterministic tool path`.

**Troubleshooting:** If `planned_tool_call` is ignored, verify `tool_name` matches a registered descriptor and risk tier is HIGH + `is_state_changing=True`. If async errors appear, re-run after the bootstrap cell; check 03 shows the `nest_asyncio` pattern.

In [3]:
import pathlib
import sys

_root = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(_root))
_contracts_src = _root / "packages" / "eXo_adapters" / "packages" / "exo-brain-core-contracts" / "src"
if _contracts_src.is_dir():
    sys.path.insert(0, str(_contracts_src))

from src.core.orchestrator import Orchestrator
from src.policies.middleware import DeterministicFirstPolicyMiddleware
from src.runtime.openai_agents_runtime import OpenAIAgentsRuntimeAdapter
from src.schemas.events import RuntimeEventType
from src.schemas.tool_io import RiskTier
from src.tools.executor import DeterministicToolExecutor
from src.tools.registry import ToolDescriptor, ToolRegistry

In [4]:
registry = ToolRegistry()
registry.register(
    ToolDescriptor(
        name="double_value",
        handler=lambda x: x * 2,
        risk_tier=RiskTier.HIGH,
        is_state_changing=True,
    )
)
policy = DeterministicFirstPolicyMiddleware()
orchestrator = Orchestrator(
    runtime_adapter=OpenAIAgentsRuntimeAdapter(),
    policy_middleware=policy,
    tool_executor=DeterministicToolExecutor(registry=registry, policy=policy),
)

context = {
    "run_id": "run_core_nb",
    "job_id": "job_core_nb",
    "task_id": "task_core_nb",
    "agent_id": "agent_core_nb",
    "planned_tool_call": {
        "call_id": "tc_core_nb",
        "tool_name": "double_value",
        "arguments": {"x": 11},
        "risk_tier": "high",
        "is_state_changing": True,
    },
}

import asyncio

async def _run_check():
    events = []
    async for event in orchestrator.run_turn("sess_core_nb", "run deterministic", context):
        events.append(event)
        print(event.event_type.value, event.payload)

    event_types = [e.event_type for e in events]
    assert RuntimeEventType.RUN_COMPLETE in event_types, "Missing RUN_COMPLETE event"
    tool_progress_states = [
        e.payload.get("state")
        for e in events
        if e.event_type == RuntimeEventType.TOOL_PROGRESS
    ]
    assert "completed" in tool_progress_states, "Missing completed TOOL_PROGRESS state"
    print("PASS: orchestrator deterministic tool path")

try:
    loop = asyncio.get_running_loop()
    import nest_asyncio
    nest_asyncio.apply()
    loop.run_until_complete(_run_check())
except RuntimeError:
    asyncio.run(_run_check())

tool_progress {'call_id': 'tc_core_nb', 'tool_name': 'double_value', 'state': 'queued', 'tool_status': '', 'error_code': '', 'job_id': '', 'lease_token': '', 'lease_expires_at_epoch': '', 'claim_attempt': ''}
tool_progress {'call_id': 'tc_core_nb', 'tool_name': 'double_value', 'state': 'running', 'tool_status': '', 'error_code': '', 'job_id': '', 'lease_token': '', 'lease_expires_at_epoch': '', 'claim_attempt': ''}
tool_progress {'call_id': 'tc_core_nb', 'tool_name': 'double_value', 'state': 'completed', 'tool_status': 'success', 'error_code': 'None', 'job_id': '', 'lease_token': '', 'lease_expires_at_epoch': '', 'claim_attempt': ''}
output_delta {'text': 'processed 1 tool result(s)'}
run_complete {'status': 'completed', 'tool_results_count': 1, 'provider_id': 'openai'}
PASS: orchestrator deterministic tool path
